# Session 26 — MLOps Pipeline for Real-Time Fraud Detection

**Goal:** the closing capstone — a full pipeline for a problem with two properties
that make it a genuinely different MLOps challenge from the rest of this course:
**severe class imbalance** (fraud is rare) and a **hard low-latency requirement**
(a real-time payment decision can't wait on a slow model).

## What's different about fraud detection

* **Imbalance**: with fraud at, say, 0.5% of transactions, a model that predicts
  "never fraud" gets 99.5% accuracy while being useless — accuracy is the wrong
  metric entirely (Session 21 touched this with precision/recall; fraud pushes it
  further).
* **Latency**: a payment authorization can't wait seconds for a model — the serving
  layer (Session 7's FastAPI pattern) needs to respond in single-digit milliseconds.
* **Adversarial drift**: unlike Session 5/17's drift (which is usually incidental),
  fraud patterns *actively* shift because fraudsters adapt to whatever the current
  model catches — monitoring (Session 5) and retraining (Session 17) matter more
  here than almost anywhere else in this course.

## Prerequisites

```bash
pip install mlflow scikit-learn fastapi
```
Runs entirely locally with a synthetic transaction dataset.

In [ ]:
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, average_precision_score

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("session26-fraud-detection")

## Step 1 — Simulate an imbalanced transaction dataset

0.6% fraud rate, with fraudulent transactions differing systematically (higher
amount, unusual hour, more distinct merchants in a short window) — mirroring how
real fraud signals look, without using any real financial data.

In [ ]:
rng = np.random.default_rng(1)
N = 50_000
FRAUD_RATE = 0.006

is_fraud = rng.random(N) < FRAUD_RATE

df = pd.DataFrame({
    "amount": np.where(is_fraud, rng.lognormal(5.5, 1.2, N), rng.lognormal(3.5, 1.0, N)),
    "hour_of_day": np.where(is_fraud, rng.choice(list(range(0, 6)) + list(range(22, 24)), N),
                             rng.integers(0, 24, N)),
    "distinct_merchants_last_hour": np.where(is_fraud, rng.poisson(4, N), rng.poisson(1, N)),
    "account_age_days": np.where(is_fraud, rng.integers(0, 60, N), rng.integers(30, 3000, N)),
    "is_international": np.where(is_fraud, rng.random(N) < 0.4, rng.random(N) < 0.03),
    "is_fraud": is_fraud.astype(int),
})

print(f"{N:,} transactions, {df['is_fraud'].sum()} fraudulent ({df['is_fraud'].mean():.3%})")

## Step 2 — Split with stratification, train with class weighting

Random splitting could leave the test set with almost no fraud examples;
`stratify=y` guarantees the same fraud rate in both splits. `class_weight="balanced"`
tells the model to weight the rare class more heavily during training, instead of
optimizing for raw accuracy on the majority class.

In [ ]:
X, y = df.drop(columns="is_fraud"), df["is_fraud"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

with mlflow.start_run(run_name="fraud_rf_balanced") as run:
    model = RandomForestClassifier(
        n_estimators=300, max_depth=10, class_weight="balanced", random_state=0,
    )
    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    avg_precision = average_precision_score(y_test, proba)  # better than AUC for rare-class problems

    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_metric("test_roc_auc", auc)
    mlflow.log_metric("test_average_precision", avg_precision)
    mlflow.sklearn.log_model(model, artifact_path="model")

    run_id = run.info.run_id
    print(f"ROC-AUC: {auc:.4f}  (looks great, but is misleading on this rare a class)")
    print(f"Average precision (PR-AUC): {avg_precision:.4f}  (the metric that actually matters here)")

## Step 3 — Why ROC-AUC is misleading here, and precision/recall isn't

With 99.4% of transactions legitimate, ROC-AUC can look excellent while the model is
still nearly useless in practice — average precision (area under the
precision-recall curve) is far more honest about performance on the rare class.

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, proba)

for target_recall in [0.5, 0.7, 0.9]:
    idx = np.argmin(np.abs(recalls[:-1] - target_recall))
    print(f"To catch {recalls[idx]:.0%} of fraud: precision={precisions[idx]:.3f} "
          f"(i.e. {precisions[idx]:.0%} of flagged transactions are actually fraud), "
          f"threshold={thresholds[idx]:.3f}")

## Step 4 — Low-latency serving

A fraud check has to run inline with the payment flow — measure the actual
prediction latency, not just correctness, since this is one of the few sessions in
this course where speed is a hard requirement, not a nice-to-have.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
import time

CHOSEN_THRESHOLD = thresholds[np.argmin(np.abs(recalls[:-1] - 0.7))]  # tuned for 70% recall

app = FastAPI(title="Real-Time Fraud Detection")

class Transaction(BaseModel):
    amount: float
    hour_of_day: int
    distinct_merchants_last_hour: int
    account_age_days: int
    is_international: bool

@app.post("/v1/score-transaction")
def score_transaction(txn: Transaction):
    row = np.array([[txn.amount, txn.hour_of_day, txn.distinct_merchants_last_hour,
                      txn.account_age_days, int(txn.is_international)]])
    proba = model.predict_proba(row)[0, 1]
    return {
        "fraud_probability": round(float(proba), 4),
        "block_transaction": bool(proba >= CHOSEN_THRESHOLD),
    }

client = TestClient(app)
sample = X_test.iloc[0].to_dict()
sample["is_international"] = bool(sample["is_international"])

start = time.perf_counter()
response = client.post("/v1/score-transaction", json=sample)
latency_ms = (time.perf_counter() - start) * 1000

print(response.status_code, response.json())
print(f"Latency: {latency_ms:.2f} ms (in-process TestClient -- a real network hop adds more)")

## Step 5 — Monitor for adversarial drift

Fraud patterns shift *because* fraudsters adapt — this makes Session 5/17's
drift-detection loop more of a hard requirement than a nice-to-have here. Simulate a
new fraud pattern emerging (fraudsters shifting toward daytime transactions to evade
the "unusual hour" signal) and confirm the drift detector catches it.

In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset

recent_fraud_mask = (y_test == 1)
new_pattern = X_test[recent_fraud_mask].copy()
new_pattern["hour_of_day"] = rng.integers(9, 17, size=len(new_pattern))  # shifted to daytime

drift_report = Report([DataDriftPreset()])
snapshot = drift_report.run(current_data=new_pattern, reference_data=X_test[recent_fraud_mask])
drift_count_metric = snapshot.dict()["metrics"][0]
n_drifted = int(drift_count_metric["value"]["count"])
drift_share = drift_count_metric["value"]["share"]

print(f"Drift detected in fraud pattern: {drift_share >= 0.5}")
print(f"Drifted columns: {n_drifted}/{len(new_pattern.columns)}")
print("\nA real system would feed this into Session 17's automatic retraining loop --")
print("the faster a shifted pattern is caught, the less fraud reaches production undetected.")

## Course wrap-up

This capstone used nearly every tool from the course: MLflow tracking (Session 1),
a FastAPI serving layer (Session 7), precision/recall analysis under class imbalance
(building on Session 21), and Evidently drift detection (Session 5) — the same
pattern repeats across Sessions 15, 16, 20, and 21's capstones, because a real
production ML system is always some combination of *train + track + validate + serve
+ monitor*, regardless of the specific domain.

## What to try next

* Add a Deepchecks (Session 11) check specifically for label leakage — a common real
  bug in fraud datasets where a post-transaction field accidentally leaks into
  training features.
* Wire the drift check in Step 5 into an automatic retraining trigger, following
  Session 17's promote-only-if-not-worse pattern exactly.
* Benchmark the FastAPI endpoint under concurrent load (`locust`) to find the actual
  throughput ceiling — a fraud-detection API's SLA is usually expressed in requests
  per second at a fixed latency budget, not just "it works."